In [ ]:
import os
import json
import sqlite3
import subprocess
import time
import re
import numpy as np
import pandas as pd

# Paths
aconsole_path = r"C:\Program Files\Aimsun\Aimsun Next 23\aconsole.exe"
base_dir = r"C:\Users\xh8\ORNL_Work\github_workspace\Aimsun-integration-to-RealTwin\Aimsun automation code"
model_dir = os.path.join(base_dir, "Chatt_test", "Model")
model_path = os.path.join(model_dir, "Chatt_test.ang")
script_dir = r"C:\Users\xh8\ORNL_Work\github_workspace\Real-Twin\realtwin\rt_aimsun"
subpath_script_path = os.path.join(script_dir, "Step7.1_SubpathGeneration.py")
assign_script_path = os.path.join(script_dir, "Step7.2_DrivingBehaviorAssign.py")

sqlite_path = os.path.join(model_dir, "Resources", "Outputs", "Chatt_test.sqlite")


parameter_csv = os.path.join(model_dir, "DrivingBehaviorParameter.csv")
subpaths_csv = os.path.join(model_dir, "subpaths.csv")
info_path = os.path.join(model_dir, "calibration_info.json")
best_txt_path = os.path.join(base_dir, "GA_DB_best.txt")

# Subpaths: (name, start_section_id, end_section_id, real_travel_time_in seconds)
SUBPATHS = [("Subpath1", 2071, 2092, 60)]
SUBPATHS = [
    ("Subpath1", 2071, 2077, 180.0),
    ("Subpath2", 2079, 2088, 180.0),
]

# Driving behavior parameters and their bounds
PARAM_NAMES = ["MinDist", "MaxAcc", "NormalDec", "MaxDec", "MinHeadway", "SensitivityFactor"]
lb = np.array([1.0, 1.5, 4.0, 5.0, 0.25, 0.0])
ub = np.array([5.0, 3.0, 5.3, 9.3, 3.0, 1.0])
num_variables = len(PARAM_NAMES)

# GA parameters
population_size = 20          # must be even
num_generations = 10
crossover_rate = 0.75
mutation_rate = 0.1

# Replication to run: auto-read from calibration_info.json, or set it manually
REPLICATION_ID = None
if REPLICATION_ID is None:
    with open(info_path) as fh:
        REPLICATION_ID = json.load(fh)["replication_id"]
replication_id = int(REPLICATION_ID)
print("replication id:", replication_id)


def run_aconsole(cmd):
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace")
    out, _ = process.communicate()
    return process.returncode, out

replication id: 2287


In [2]:
SUBPATHS

for name, _, _, real_tt in SUBPATHS:
    print(f"Subpath: {name}, Real travel time: {real_tt} seconds")

print("Configured subpath endpoints:")
for name, start_id, end_id, _ in SUBPATHS:
    print(f"  {name}: {start_id} -> {end_id}")

Subpath: Subpath1, Real travel time: 60 seconds
Configured subpath endpoints:
  Subpath1: 2071 -> 2092


In [3]:
pd.DataFrame(SUBPATHS).to_csv(subpaths_csv, index=False, header=False)
rc, out = run_aconsole([aconsole_path, "-script", subpath_script_path, model_path])
print(out)

SUBPATH_IDS = dict(re.findall(r"SUBPATH_ID (\S+)=(\d+)", out))
subpath_targets = []
for name, start_id, end_id, real_tt in SUBPATHS:
    if name not in SUBPATH_IDS:
        raise RuntimeError(
            "subpath '%s' was not created for endpoints %s -> %s; "
            "use valid GKSection IDs from the loaded Aimsun model"
            % (name, start_id, end_id))
    subpath_targets.append((name, int(SUBPATH_IDS[name]), float(real_tt)))
print("subpath targets (name, aimsun id, real travel time s):")
for t in subpath_targets:
    print("  ", t)

[2026-09-09 16:14:16.610] [info] Command: [C:\Program Files\Aimsun\Aimsun Next 23\aconsole.exe][-script][C:\Users\xh8\ORNL_Work\github_workspace\Real-Twin\realtwin\rt_aimsun\Step7.1_SubpathGeneration.py][C:\Users\xh8\ORNL_Work\github_workspace\Aimsun-integration-to-RealTwin\Aimsun automation code\Chatt_test\Model\Chatt_test.ang]
[2026-09-09 16:14:16.751] [info] Platform License
[2026-09-09 16:14:16.767] [info] License Keys to consider: 548880995
[2026-09-09 16:14:16.771] [info] Aimsun Console instance id {efc8048d-3916-4d1e-beff-187c94a91c29}
[2026-09-09 16:14:16.771] [info] Aimsun Console PID 26096
[2026-09-09 16:14:16.772] [info] Qt Home C:/Program Files/Aimsun/Aimsun Next 23
[2026-09-09 16:14:16.848] [warning] Alerter not started, missing key.
[2026-09-09 16:14:16.860] [info] USING PROJECT DB QSQLITE  C:\Users\xh8\AppData\Roaming\Aimsun\Aimsun Next\23.0.0/aimsun.db USER 
[2026-09-09 16:14:17.046] [info] Application Locale en_US, translation en_US
[2026-09-09 16:14:17.412] [info] AVC

In [4]:
def newParameter(scaled):
    """Apply a scaled parameter vector to the Car vehicle in the model.

    aconsole may crash on exit after finishing (0xC0000374), so success is
    judged by the ':Applied' confirmation line, not the return code.
    """
    np.savetxt(parameter_csv, np.asarray(scaled, dtype=float), delimiter=",", fmt="%s")
    rc, out = run_aconsole([aconsole_path, "-script", assign_script_path, model_path])
    if ":Applied" not in out:
        print(out)
        raise RuntimeError("driving behavior assignment failed (return code %s)" % rc)


def runAimsun():
    """Clear this replication's old results, then execute it.

    Success is judged by fresh MISECT rows for this replication appearing in
    the sqlite output, not by the aconsole return code.
    """
    if os.path.exists(sqlite_path):
        try:
            con = sqlite3.connect(sqlite_path)
            for table in ("MISUBPATH", "MISECT"):
                try:
                    con.execute("DELETE FROM %s WHERE did = ?" % table, (replication_id,))
                except sqlite3.Error:
                    pass
            con.commit()
            con.close()
        except sqlite3.Error:
            pass
    rc, out = run_aconsole([aconsole_path, "--project", model_path,
                            "--command", "execute", "--target", str(replication_id)])
    n_rows = 0
    if os.path.exists(sqlite_path):
        try:
            con = sqlite3.connect(sqlite_path)
            n_rows = con.execute("SELECT COUNT(*) FROM MISECT WHERE did = ?",
                                 (replication_id,)).fetchone()[0]
            con.close()
        except sqlite3.Error:
            pass
    if n_rows == 0:
        print(out)
        raise RuntimeError("simulation produced no results (return code %s)" % rc)


def resultFitness():
    """MAE between simulated and real subpath travel times (whole period).

    Returns (None, sim_tts) when any subpath has no usable travel time --
    e.g. the parameter set gridlocked the network so no vehicle completed the
    subpath within the simulation. The caller applies a penalty fitness.
    """
    con = sqlite3.connect(sqlite_path)
    sub = pd.read_sql_query(
        "SELECT oid, ttime FROM MISUBPATH WHERE did = %d AND sid = 0 AND ent = 0"
        % replication_id, con)
    con.close()
    sub = sub.drop_duplicates(subset="oid", keep="last").set_index("oid")
    sim_tts = {}
    errors = []
    degenerate = False
    for name, sp_id, real_tt in subpath_targets:
        if sp_id not in sub.index:
            sim_tts[name] = None
            degenerate = True
            continue
        tt = float(sub.loc[sp_id, "ttime"])
        if tt <= 0:                      # Aimsun writes -1 / 0 when no vehicle finished
            sim_tts[name] = None
            degenerate = True
            continue
        sim_tts[name] = tt
        errors.append(abs(float(real_tt) - tt))
    if degenerate or not errors:
        return None, sim_tts
    MAE = float(np.mean(errors))
    return MAE, sim_tts


def resultAnalysis():
    """Mean approach-level GEH (optional check; needs calibration_info.json)."""
    if not os.path.exists(info_path):
        return None, None
    with open(info_path) as fh:
        field_df = pd.DataFrame(json.load(fh)["field_approaches"]).rename(
            columns={"count": "realcount"})
    con = sqlite3.connect(sqlite_path)
    section = pd.read_sql_query(
        "SELECT oid, count FROM MISECT WHERE did = %d AND sid = 0 AND ent = 0"
        % replication_id, con)
    con.close()
    section = section.drop_duplicates(subset="oid", keep="last")
    compare = field_df.merge(section, left_on="section", right_on="oid", how="left")
    compare = compare.dropna(subset=["count"])
    compare["GEH"] = np.sqrt(2 * ((compare["count"] - compare["realcount"]) ** 2)
                             / (compare["count"] + compare["realcount"]))
    return compare["GEH"].mean(), (compare["GEH"] < 5).mean()

In [5]:
# Optional smoke test: one evaluation with mid-range parameters
test_scaled = lb + 0.5 * (ub - lb)
print("testing parameters:", dict(zip(PARAM_NAMES, np.round(test_scaled, 3))))
newParameter(test_scaled)
runAimsun()
MAE, sim_tts = resultFitness()
for name, sp_id, real_tt in subpath_targets:
    sim = sim_tts.get(name)
    print("%s (id %s): simulated %s vs real %.1f s"
          % (name, sp_id, "%.1f s" % sim if sim is not None else "no data", real_tt))
print("MAE = %s" % ("%.3f s" % MAE if MAE is not None else "degenerate (no travel time)"))

testing parameters: {'MinDist': np.float64(3.0), 'MaxAcc': np.float64(2.25), 'NormalDec': np.float64(4.65), 'MaxDec': np.float64(7.15), 'MinHeadway': np.float64(1.625), 'SensitivityFactor': np.float64(0.5)}
Subpath1 (id 2304): simulated 15.9 s vs real 60.0 s
MAE = 44.059 s


In [6]:
from mealpy import FloatVar, GA

PENALTY_MAE = 1.0e4

eval_count = [0]
best_so_far = [np.inf]

def objective_function(x):
    x_scaled = lb + np.asarray(x, dtype=float) * (ub - lb)
    newParameter(x_scaled)
    runAimsun()
    value, sim_tts = resultFitness()
    eval_count[0] += 1
    if value is None:
        missing = [n for n, tt in sim_tts.items() if tt is None]
        print(f"  :evaluation {eval_count[0]}: degenerate simulation (no travel time for {', '.join(missing)}) - penalty applied")
        # raise Exception(f"Degenerate simulation: no demands for {missing}, add demand or change different path")
        return PENALTY_MAE
    # Report and save only when a better solution is found
    if value < best_so_far[0]:
        best_so_far[0] = value
        print(f"  :evaluation {eval_count[0]}: new best MAE = {value:.3f} s   {dict(zip(PARAM_NAMES, np.round(x_scaled, 4)))}")
        np.savetxt(best_txt_path, x_scaled, fmt="%f",
                   header=f"MAE = %.6f s (evaluation {eval_count[0]})  order: {', '.join(PARAM_NAMES)}")
    return value


problem_dict = {
    "obj_func": objective_function,
    "bounds": FloatVar(lb=[0.0] * num_variables, ub=[1.0] * num_variables),
    "minmax": "min",
    "verbose": True,
    "save_population": True,
}

optimizer = GA.BaseGA(
    epoch=num_generations,
    pop_size=population_size,
    pc=crossover_rate,
    pm=mutation_rate,
)

start_time = time.time()
g_best = optimizer.solve(problem_dict, mode="single")
print("elapsed: %.1f min" % ((time.time() - start_time) / 60))

best_scaled = lb + np.asarray(g_best.solution, dtype=float) * (ub - lb)
print("Best parameters:", dict(zip(PARAM_NAMES, np.round(best_scaled, 4))))
print("Best MAE:", g_best.target.fitness)
np.savetxt(best_txt_path, best_scaled, fmt="%f",
           header=f"MAE = %.6f s (final)  order: {', '.join(PARAM_NAMES)}")

2026/09/09 04:14:26 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: BaseGA(epoch=10, pop_size=20, pc=0.75, pm=0.1)


  :evaluation 1: new best MAE = 43.136 s   {'MinDist': np.float64(4.1055), 'MaxAcc': np.float64(1.6607), 'NormalDec': np.float64(4.1532), 'MaxDec': np.float64(8.0754), 'MinHeadway': np.float64(1.0435), 'SensitivityFactor': np.float64(0.103)}
  :evaluation 3: new best MAE = 42.589 s   {'MinDist': np.float64(4.8434), 'MaxAcc': np.float64(2.324), 'NormalDec': np.float64(4.9325), 'MaxDec': np.float64(8.8239), 'MinHeadway': np.float64(2.6887), 'SensitivityFactor': np.float64(0.6575)}


KeyboardInterrupt: 

In [ ]:
# Final run with the best parameters: they stay applied in the .ang.
# If the GA was interrupted, the best-so-far is recovered from GA_DB_best.txt.
try:
    best_scaled
except NameError:
    best_scaled = np.loadtxt(best_txt_path)
    print("loaded best parameters from GA_DB_best.txt")

newParameter(best_scaled)
runAimsun()
MAE, sim_tts = resultFitness()
print("Best parameters:", dict(zip(PARAM_NAMES, np.round(best_scaled, 4))))
print("Final MAE = %s" % ("%.3f s" % MAE if MAE is not None else "degenerate"))
for name, sp_id, real_tt in subpath_targets:
    sim = sim_tts.get(name)
    print("  %s (id %s): calibrated %s vs real %.1f s"
          % (name, sp_id, "%.1f s" % sim if sim is not None else "no data", real_tt))

meanGEH, GEHPercent = resultAnalysis()
if meanGEH is not None:
    print("Final mean GEH = %.3f (%.2f%% of approaches < 5)"
          % (meanGEH, GEHPercent * 100))